# 🧠 Train Sparse Autoencoders on nanochat (Google Colab T4)

This notebook trains Sparse Autoencoders (SAEs) on a pre-trained nanochat model using Google Colab's **free tier T4 GPU**.

## What You'll Do:
- ✅ Load a pre-trained nanochat checkpoint (d20, 561M params)
- ✅ Collect activations from your custom reference dataset
- ✅ Train SAEs to discover interpretable features
- ✅ Visualize what your model learned
- ✅ Save checkpoints to Google Drive

## Before You Start:
1. **Enable T4 GPU**: Runtime → Change runtime type → T4 GPU
2. **Mount Google Drive**: For checkpointing (optional but recommended)
3. **Estimated Time**: 1-2 hours per layer on T4

---

## 📦 1. Environment Setup

First, let's verify we have a GPU and install dependencies.

In [ ]:
# Check GPU availability
import torch
import subprocess

print("🔍 Checking GPU...")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Found: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    
    if "T4" in gpu_name:
        print("   Perfect! T4 GPU is ideal for this notebook.")
    else:
        print(f"   Note: This notebook is optimized for T4, but {gpu_name} should work too.")
else:
    print("❌ No GPU found!")
    print("   Go to: Runtime → Change runtime type → Select 'T4 GPU'")
    raise RuntimeError("GPU required for SAE training")

print(f"\n🐍 Python: {subprocess.check_output(['python', '--version']).decode().strip()}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"💻 CUDA: {torch.version.cuda}")

In [ ]:
%%bash
# Install system dependencies for Rust (needed for tokenizer)
echo "📥 Installing system dependencies..."
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
source "$HOME/.cargo/env"

In [ ]:
%%bash
# Clone the nanochat-SAE repository
echo "📥 Cloning nanochat-SAE repository..."
if [ ! -d "nanochat-SAE" ]; then
    git clone https://github.com/SolshineCode/nanochat-SAE.git
    cd nanochat-SAE
else
    echo "Repository already cloned."
    cd nanochat-SAE
    git pull
fi

In [ ]:
%%bash
cd nanochat-SAE

# Install uv (fast Python package manager)
echo "📥 Installing uv package manager..."
curl -LsSf https://astral.sh/uv/install.sh | sh
source "$HOME/.cargo/env"

# Install dependencies with GPU support
echo "📦 Installing Python dependencies (this may take a few minutes)..."
uv venv --python 3.10
source .venv/bin/activate
uv pip install --extra-index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
uv pip install datasets numpy regex setuptools tiktoken tokenizers wandb psutil

In [ ]:
%%bash
cd nanochat-SAE

# Build Rust tokenizer
echo "🔧 Building Rust tokenizer..."
source "$HOME/.cargo/env"
source .venv/bin/activate

# Install maturin
uv pip install maturin

# Build the tokenizer
maturin develop --release --manifest-path rustbpe/Cargo.toml

echo "✅ Setup complete!"

In [ ]:
# Change to the repo directory and import modules
import os
import sys

# Add to path
sys.path.insert(0, '/content/nanochat-SAE')
os.chdir('/content/nanochat-SAE')

# Now import nanochat modules
import torch
import torch.nn.functional as F
from pathlib import Path
import json
import numpy as np
from tqdm.auto import tqdm

print("✅ All imports successful!")

## 💾 2. Mount Google Drive (Optional but Recommended)

Mount your Google Drive to save checkpoints and results. This prevents losing progress if the Colab session disconnects.

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create directories for checkpoints and results
DRIVE_DIR = Path('/content/drive/MyDrive/nanochat-SAE')
CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints'
RESULTS_DIR = DRIVE_DIR / 'results'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Google Drive mounted!")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
print(f"   Results: {RESULTS_DIR}")

## 🤖 3. Download Pre-trained nanochat Model

We'll use a pre-trained d20 model (561M parameters). You can either:
1. **Download from a public URL** (if available)
2. **Upload your own checkpoint** to Google Drive
3. **Train a tiny model** for testing (not recommended for real use)

In [ ]:
# Configuration
MODEL_PATH = None  # Set this if you have a specific checkpoint

# Option 1: Upload your own checkpoint from Google Drive
# Uncomment and set the path if you have a checkpoint in Drive:
# MODEL_PATH = '/content/drive/MyDrive/nanochat-SAE/checkpoints/base_final.pt'

# Option 2: Download from URL (if you have a public checkpoint URL)
# MODEL_URL = 'https://example.com/nanochat_d20.pt'  # Replace with actual URL

# For this demo, we'll show how to load the model structure
# In practice, you need a real checkpoint from training or download

print("📝 Model Configuration:")
if MODEL_PATH and Path(MODEL_PATH).exists():
    print(f"   Using checkpoint: {MODEL_PATH}")
else:
    print("   ⚠️  No checkpoint specified.")
    print("   You need to either:")
    print("   1. Upload a checkpoint to Google Drive and set MODEL_PATH")
    print("   2. Train a model first using speedrun.sh")
    print("   3. Download from a public URL if available")

In [ ]:
# Load the model
from nanochat.gpt import GPT, GPTConfig

def load_nanochat_model(checkpoint_path, device='cuda'):
    """Load a nanochat model from checkpoint."""
    print(f"📂 Loading model from {checkpoint_path}...")
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Extract config
    config_dict = checkpoint.get('config', {})
    
    # Create model config
    config = GPTConfig(
        sequence_len=config_dict.get('sequence_len', 1024),
        vocab_size=config_dict.get('vocab_size', 50304),
        n_layer=config_dict.get('n_layer', 20),
        n_head=config_dict.get('n_head', 10),
        n_kv_head=config_dict.get('n_kv_head', 10),
        n_embd=config_dict.get('n_embd', 1280),
    )
    
    # Create and load model
    model = GPT(config)
    model.load_state_dict(checkpoint['model'], strict=False)
    model.to(device)
    model.eval()
    
    num_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"✅ Model loaded: {num_params:.1f}M parameters")
    print(f"   Layers: {config.n_layer}")
    print(f"   Embedding dim: {config.n_embd}")
    print(f"   Vocab size: {config.vocab_size}")
    
    return model, config

# Load the model if checkpoint exists
if MODEL_PATH and Path(MODEL_PATH).exists():
    model, model_config = load_nanochat_model(MODEL_PATH)
    print("\n🎉 Model ready for SAE training!")
else:
    print("⚠️  Skipping model load - no checkpoint available")
    print("   Please upload a checkpoint or download one first.")

## 📊 4. Prepare Your Reference Dataset

Load your custom reference dataset for collecting activations. Supports:
- Plain text files (.txt)
- JSONL files with text field
- Hugging Face datasets

In [ ]:
# Option 1: Upload a text file from your computer
from google.colab import files

print("📤 Upload your reference dataset (optional):")
print("   Supported formats: .txt, .jsonl")
print("   Or skip and use random data for testing")

# Uncomment to upload:
# uploaded = files.upload()

# Option 2: Load from Google Drive
# DATASET_PATH = '/content/drive/MyDrive/nanochat-SAE/datasets/my_data.txt'

# Option 3: Use a Hugging Face dataset
# from datasets import load_dataset
# dataset = load_dataset('wikipedia', '20220301.en', split='train[:1000]')

In [ ]:
def load_reference_dataset(path=None, max_samples=10000):
    """Load reference dataset for activation collection."""
    
    if path is None:
        print("⚠️  No dataset provided, will use random tokens for demo")
        return None
    
    path = Path(path)
    
    if not path.exists():
        print(f"❌ Dataset not found: {path}")
        return None
    
    print(f"📂 Loading dataset from {path}...")
    
    texts = []
    
    if path.suffix == '.txt':
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
            # Split into chunks
            chunks = [text[i:i+512] for i in range(0, len(text), 512)]
            texts.extend(chunks[:max_samples])
    
    elif path.suffix == '.jsonl':
        with open(path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= max_samples:
                    break
                data = json.loads(line)
                # Assume 'text' field, adjust as needed
                texts.append(data.get('text', ''))
    
    else:
        print(f"❌ Unsupported format: {path.suffix}")
        return None
    
    print(f"✅ Loaded {len(texts)} text samples")
    return texts

# Load dataset (set path if you have one)
DATASET_PATH = None  # Set to your dataset path
reference_texts = load_reference_dataset(DATASET_PATH)

if reference_texts:
    print(f"\n📝 Sample text (first 200 chars):")
    print(reference_texts[0][:200] + "...")

## 🧠 5. Train Sparse Autoencoder (T4 Optimized)

Now we'll train an SAE on activations from your model. Settings are optimized for T4 GPU constraints.

In [ ]:
# T4-Optimized SAE Configuration
SAE_CONFIG = {
    # Which layer to analyze
    'layer': 10,  # Middle layer of d20 model (0-19)
    
    # SAE architecture
    'expansion_factor': 4,  # 4x expansion (conservative for T4)
    'activation': 'topk',   # topk, relu, or gated
    'k': 32,                # Number of active features (for topk)
    
    # Data collection
    'num_activations': 100_000,  # Reduced for T4 (original: 1M)
    'sequence_length': 512,       # Shorter sequences save memory
    'collect_batch_size': 4,      # Small batches during collection
    
    # Training
    'train_batch_size': 512,      # Training batch size
    'num_epochs': 5,              # Fewer epochs for faster iteration
    'learning_rate': 3e-4,
    'weight_decay': 0.0,
    
    # Checkpointing
    'checkpoint_every': 1000,     # Save every N steps
    'validate_every': 500,        # Validate every N steps
}

print("⚙️  SAE Configuration (T4-Optimized):")
print(json.dumps(SAE_CONFIG, indent=2))

# Calculate SAE size
if 'model_config' in locals():
    d_in = model_config.n_embd
    d_sae = d_in * SAE_CONFIG['expansion_factor']
    print(f"\n📐 SAE Dimensions:")
    print(f"   Input: {d_in}")
    print(f"   SAE features: {d_sae}")
    print(f"   Active features: {SAE_CONFIG['k']}")

In [ ]:
# Collect activations from the model
from sae.hooks import ActivationCollector
from nanochat.tokenizer import get_tokenizer

def collect_activations_from_dataset(
    model,
    layer_idx,
    num_activations,
    reference_texts=None,
    sequence_length=512,
    batch_size=4,
    device='cuda'
):
    """Collect activations from model using reference dataset."""
    
    hook_point = f"blocks.{layer_idx}.hook_resid_post"
    print(f"🎯 Collecting activations from {hook_point}...")
    print(f"   Target: {num_activations:,} activations")
    
    # Setup activation collector
    collector = ActivationCollector(
        model=model,
        hook_points=[hook_point],
        max_activations=num_activations,
        device='cpu',  # Store on CPU to save GPU memory
    )
    
    # Load tokenizer
    tokenizer = get_tokenizer()
    
    model.eval()
    with torch.no_grad(), collector:
        num_batches = (num_activations // (sequence_length * batch_size)) + 1
        
        with tqdm(total=num_activations, desc="Collecting") as pbar:
            for batch_idx in range(num_batches):
                # Get batch of tokens
                if reference_texts:
                    # Use reference dataset
                    start_idx = (batch_idx * batch_size) % len(reference_texts)
                    batch_texts = reference_texts[start_idx:start_idx + batch_size]
                    
                    # Tokenize
                    tokens_list = []
                    for text in batch_texts:
                        toks = tokenizer.encode(text[:sequence_length * 4])  # Rough char estimate
                        toks = toks[:sequence_length]  # Truncate
                        # Pad if needed
                        if len(toks) < sequence_length:
                            toks.extend([0] * (sequence_length - len(toks)))
                        tokens_list.append(toks)
                    
                    tokens = torch.tensor(tokens_list, device=device)
                else:
                    # Use random tokens for testing
                    tokens = torch.randint(
                        0,
                        model.config.vocab_size,
                        (batch_size, sequence_length),
                        device=device
                    )
                
                # Forward pass
                _ = model(tokens)
                
                # Update progress
                current_count = collector.counts[hook_point]
                pbar.update(current_count - pbar.n)
                
                # Check if done
                if current_count >= num_activations:
                    break
    
    # Get collected activations
    activations = collector.get_activations()[hook_point]
    print(f"\n✅ Collected {activations.shape[0]:,} activations")
    print(f"   Shape: {activations.shape}")
    print(f"   Memory: {activations.nbytes / 1e9:.2f} GB")
    
    return activations

# Collect activations (only if model is loaded)
if 'model' in locals():
    activations = collect_activations_from_dataset(
        model=model,
        layer_idx=SAE_CONFIG['layer'],
        num_activations=SAE_CONFIG['num_activations'],
        reference_texts=reference_texts,
        sequence_length=SAE_CONFIG['sequence_length'],
        batch_size=SAE_CONFIG['collect_batch_size'],
    )
    
    # Save activations to Drive (optional)
    if 'CHECKPOINT_DIR' in locals():
        acts_path = CHECKPOINT_DIR / f"activations_layer{SAE_CONFIG['layer']}.pt"
        torch.save(activations, acts_path)
        print(f"💾 Saved activations to {acts_path}")
else:
    print("⚠️  Skipping activation collection - no model loaded")

In [ ]:
# Train the SAE
from sae.config import SAEConfig
from sae.trainer import train_sae_from_activations
from sae.runtime import save_sae

if 'activations' in locals() and 'model_config' in locals():
    # Create SAE config
    sae_config = SAEConfig(
        d_in=model_config.n_embd,
        hook_point=f"blocks.{SAE_CONFIG['layer']}.hook_resid_post",
        expansion_factor=SAE_CONFIG['expansion_factor'],
        activation=SAE_CONFIG['activation'],
        k=SAE_CONFIG['k'],
        num_activations=SAE_CONFIG['num_activations'],
        batch_size=SAE_CONFIG['train_batch_size'],
        num_epochs=SAE_CONFIG['num_epochs'],
        learning_rate=SAE_CONFIG['learning_rate'],
    )
    
    print("🚀 Starting SAE training...")
    print(f"   Architecture: {sae_config.activation.upper()}")
    print(f"   Features: {sae_config.d_sae:,}")
    print(f"   Active k: {sae_config.k}")
    print(f"   Batch size: {sae_config.batch_size}")
    print(f"   Epochs: {sae_config.num_epochs}")
    
    # Create output directory
    if 'RESULTS_DIR' in locals():
        output_dir = RESULTS_DIR / f"layer_{SAE_CONFIG['layer']}"
    else:
        output_dir = Path('/content/sae_outputs') / f"layer_{SAE_CONFIG['layer']}"
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Train SAE
    sae, trainer = train_sae_from_activations(
        activations=activations,
        config=sae_config,
        device='cuda',
        save_dir=output_dir,
        verbose=True,
    )
    
    # Save final model
    final_path = output_dir / 'sae_final.pt'
    save_sae(
        sae=sae,
        config=sae_config,
        save_path=final_path,
        training_steps=trainer.step,
        best_val_loss=trainer.best_val_loss,
    )
    
    print(f"\n🎉 SAE training complete!")
    print(f"   Model saved to: {final_path}")
    print(f"   Training steps: {trainer.step:,}")
    print(f"   Best val loss: {trainer.best_val_loss:.4f}")
else:
    print("⚠️  Skipping SAE training - no activations collected")

## 📊 6. Visualize Learned Features

Let's explore what features the SAE discovered!

In [ ]:
# Evaluate SAE quality
if 'sae' in locals() and 'activations' in locals():
    print("📊 Evaluating SAE quality...\n")
    
    # Sample evaluation
    sae.eval()
    with torch.no_grad():
        # Take a subset for evaluation
        eval_acts = activations[:10000].to('cuda')
        
        # Forward pass through SAE
        reconstructed, feature_acts = sae(eval_acts)
        
        # Compute metrics
        mse = F.mse_loss(reconstructed, eval_acts)
        l0 = (feature_acts != 0).float().sum(dim=-1).mean()
        
        # Explained variance
        total_var = eval_acts.var()
        residual_var = (eval_acts - reconstructed).var()
        explained_var = 1 - (residual_var / total_var)
        
        print(f"📈 SAE Quality Metrics:")
        print(f"   MSE Loss: {mse.item():.6f}")
        print(f"   L0 (avg active): {l0.item():.1f}")
        print(f"   Explained Variance: {explained_var.item():.1%}")
        
        # Dead features
        feature_max_acts = feature_acts.abs().max(dim=0)[0]
        dead_features = (feature_max_acts == 0).sum()
        dead_pct = 100 * dead_features / len(feature_max_acts)
        print(f"   Dead Features: {dead_features}/{len(feature_max_acts)} ({dead_pct:.1f}%)")
        
        # Most active features
        feature_freq = (feature_acts != 0).float().mean(dim=0)
        top_features = feature_freq.topk(10)
        
        print(f"\n🔥 Top 10 Most Active Features:")
        for i, (freq, idx) in enumerate(zip(top_features.values, top_features.indices)):
            print(f"   {i+1}. Feature {idx.item()}: {freq.item():.1%} activation rate")
else:
    print("⚠️  Skipping evaluation - no trained SAE")

In [ ]:
# Visualize feature activation patterns
import matplotlib.pyplot as plt

if 'sae' in locals() and 'feature_acts' in locals():
    print("🎨 Visualizing feature patterns...\n")
    
    # Plot activation frequency distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Feature activation frequency
    feature_freq = (feature_acts != 0).float().mean(dim=0).cpu().numpy()
    axes[0].hist(feature_freq, bins=50, edgecolor='black')
    axes[0].set_xlabel('Activation Frequency')
    axes[0].set_ylabel('Number of Features')
    axes[0].set_title('Feature Activation Frequency Distribution')
    axes[0].axvline(feature_freq.mean(), color='red', linestyle='--', 
                    label=f'Mean: {feature_freq.mean():.3f}')
    axes[0].legend()
    
    # Feature activation magnitudes
    feature_magnitudes = feature_acts.abs().mean(dim=0).cpu().numpy()
    axes[1].hist(feature_magnitudes, bins=50, edgecolor='black')
    axes[1].set_xlabel('Mean Activation Magnitude')
    axes[1].set_ylabel('Number of Features')
    axes[1].set_title('Feature Activation Magnitude Distribution')
    axes[1].axvline(feature_magnitudes.mean(), color='red', linestyle='--',
                    label=f'Mean: {feature_magnitudes.mean():.3f}')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Save figure
    if 'output_dir' in locals():
        fig.savefig(output_dir / 'feature_distribution.png', dpi=150, bbox_inches='tight')
        print(f"💾 Saved visualization to {output_dir / 'feature_distribution.png'}")
else:
    print("⚠️  Skipping visualization - no feature activations")

## 🎯 Next Steps

Congratulations! You've trained a Sparse Autoencoder on nanochat. Here's what you can do next:

### 1. Analyze Features
- Run `scripts/sae_viz.py` to generate interactive feature dashboards
- Identify interpretable features (negation, sentiment, entities, etc.)
- Find which inputs maximally activate each feature

### 2. Feature Steering
- Use `sae.runtime.InterpretableModel` to steer model behavior
- Amplify or suppress specific features during generation
- Test how features affect model outputs

### 3. Multi-Layer Analysis
- Train SAEs on different layers (early vs. late)
- Compare features across layers
- Discover feature composition and circuits

### 4. Share Your Findings
- Upload to Neuronpedia for community exploration
- Write about your discoveries
- Contribute back to the repo!

### 5. Scale Up
- Try larger models (d26, d30)
- Use Colab Pro for better GPUs (A100, V100)
- Train with more activations and larger expansion factors

---

## 📚 Resources

- [nanochat-SAE GitHub](https://github.com/SolshineCode/nanochat-SAE)
- [nanochat Original](https://github.com/karpathy/nanochat)
- [Anthropic: Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features)
- [OpenAI: Scaling SAEs](https://openai.com/research/sparse-autoencoders)
- [Neuronpedia](https://neuronpedia.org)

---

**Questions or issues?** Open an issue on GitHub or reach out to the community!

**Found something cool?** Share your discoveries! Tweet @karpathy 🎉

In [ ]:
# Print summary
print("="*80)
print("🎉 TRAINING COMPLETE!")
print("="*80)

if 'sae' in locals():
    print(f"\n✅ Successfully trained SAE on layer {SAE_CONFIG['layer']}")
    print(f"   Features: {sae_config.d_sae:,}")
    print(f"   Activations used: {SAE_CONFIG['num_activations']:,}")
    
    if 'output_dir' in locals():
        print(f"\n💾 Files saved to:")
        print(f"   {output_dir}")
        print(f"\n📂 Contents:")
        for file in output_dir.iterdir():
            size_mb = file.stat().st_size / 1e6
            print(f"   - {file.name} ({size_mb:.1f} MB)")
    
    print(f"\n🚀 Next: Explore features with sae_viz.py or try feature steering!")
else:
    print("\n⚠️  Training not completed. Check the cells above for errors.")
    print("   Make sure you have:")
    print("   1. Enabled T4 GPU")
    print("   2. Loaded a model checkpoint")
    print("   3. Collected activations")

print("\n" + "="*80)